In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
import random
import numpy as np
import re
import os
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device: {}".format(device))
torch.set_grad_enabled(False)

save_path = "../data/"

if not os.path.exists(save_path + "embedding/"):
        os.makedirs(save_path + "embedding/")


In [ ]:
def setup(model_name, texts):
    
    # retrieving model and tokenizer from huggingface transformers
    tokenizer = AutoTokenizer.from_pretrained(model_name, model_max_length = 256)
    model = AutoModel.from_pretrained(model_name, output_hidden_states = True).to(device)
    model.eval()
        
    # defining the layers to be extracted
    if "distilbert" in model_name:
        layers = [0,2, 4, 6]
    elif "large" in model_name:
        layers = [0, 6, 12, 18, 24]
    else:
        layers = [0, 3, 6, 9, 12]
        
    # substitute mask token if necessary
    if tokenizer.mask_token != "[MASK]":
        texts = [re.sub(r"\[MASK\]", tokenizer.mask_token, text) for text in texts]
    
    mask_idx = tokenizer.get_vocab()[tokenizer.mask_token]
    
    print("-"*100)
    print("Setting up model {}".format(model_name))
    print("mask token: {}, mask index: {}".format(tokenizer.mask_token, mask_idx))
    rand_int = random.randint(0, len(texts)-1)
    print("example tokenization: " + texts[rand_int])
    print([tokenizer.decode([token]) for token in tokenizer(texts[rand_int], add_special_tokens = True)["input_ids"]])
    print("- "*51)
    
    return model, tokenizer, texts, mask_idx, layers


## Getting the data

In [ ]:
dataset_name = "cctweets-random" # cctweets-random # semcor&omsti # cctweets-activist

df = pd.read_csv(f"{save_path}/{dataset_name}_df.csv", index_col = 0)

texts = list(df["text_masked"])
ids = list(df["id"])

# test
model, tokenizer, texts_, mask_idx, layers = setup("microsoft/deberta-base", texts)

## Models

In [ ]:
models = ['bert-base-uncased', 'microsoft/deberta-base']

## Extracting Representations

In [20]:
batch_size= 128
n_batches = len(texts) // batch_size +1

for model_name in models:
    
    torch.cuda.reset_peak_memory_stats()
    
    # setting up the model
    model, tokenizer, texts_, mask_idx, layers = setup(model_name, texts)
        
    # prepping dictionary of representations and list of ids
    embeddings = {}
    for layer in layers:
        embeddings[layer] = torch.empty(0, model.config.hidden_size)
    embedding_ids = []
    
    # looping over data
    for batch, batch_idx in enumerate(range(0, n_batches*batch_size, batch_size)):

        # model throughput
        inputs = tokenizer(texts_[batch_idx:batch_idx+batch_size], return_tensors = "pt", padding = True, truncation = True).to(device)
        mask = (inputs["input_ids"] == mask_idx).unsqueeze(2).to(device)
        
        outputs = model(**inputs)
        
        # identifying the layer embeddings in the output tuple
        output_by_layer = [i for i in range(0, len(outputs)) if len(outputs[i]) == max(layers)+1]
        assert len(output_by_layer) == 1, "more than one output for all layers"
        l = output_by_layer[0]

        # extracting [MASK] representations
        for layer in layers:
            
            e = torch.masked_select(outputs[l][layer], mask).view(-1, model.config.hidden_size).to("cpu")

            embeddings[layer] = torch.cat((embeddings[layer], e), dim = 0) #append new representations
        
        # collecting ids
        batch_tweet_ids = ids[batch_idx:batch_idx+batch_size]
        for i in range(0, inputs["input_ids"].shape[0]):
            embedding_ids.extend([batch_tweet_ids[i]] *torch.sum(mask[i]).item())
            
    
    # assertions
    assert embedding_ids == ids, "Inconsistent ids"
    
    for layer in layers:
        assert embeddings[layer].shape[0] == len(embedding_ids), "Number of embeddings not equal to ids"
   
    # saving the representations
    for layer in layers:
        np.save(f'{save_path}embedding/{dataset_name}_{re.sub("/", "-", model_name)}_layer{str(layer)}', embeddings[layer])
            
    print("max {} GiB GPU memory allocated with model {}".format(round(torch.cuda.max_memory_allocated()/1e9, 2), model_name))


max 0.0 GiB GPU memory allocated with model microsoft/deberta-base
